In [ ]:
# ============================================================================
# 脑MRI体素分类 - TabNet迁移学习 (完整严谨版本)
# 主要改进：
# 1. 输入维度适配：341 -> 245
# 2. 正确处理分类特征
# 3. 支持entmax激活函数
# 4. 修复测试数据加载器问题
# 5. 改进权重加载逻辑
# ============================================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import pytorch_lightning as pl
from pytorch_lightning.callbacks import Callback, ModelCheckpoint, EarlyStopping, LearningRateMonitor
from torch.utils.data import DataLoader, TensorDataset
import h5py
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, KBinsDiscretizer
from sklearn.metrics import f1_score
import json
import os
from collections import OrderedDict
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import pandas as pd
from typing import Dict, List, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

# 设置随机种子
pl.seed_everything(42)

# ============================================================================
# Part 1: 数据模块 - 包含标准化和特征适配
# ============================================================================

class BrainVoxelDataModule(pl.LightningDataModule):
    """
    数据模块 - 严格实现数据加载、标准化和特征适配
    """
    
    def __init__(self, 
                 data_path: str, 
                 batch_size: int = 256, 
                 num_workers: int = 4,
                 adapt_features: bool = True):
        super().__init__()
        self.data_path = data_path
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.adapt_features = adapt_features
        self.scaler = StandardScaler()
        self.discretizer = None
        
    def setup(self, stage: Optional[str] = None):
        """加载数据并进行标准化"""
        print("="*60)
        print("📂 数据加载与预处理")
        print("="*60)
        
        # 加载原始数据
        with h5py.File(self.data_path, 'r') as f:
            train_data = np.array(f['data']).transpose()
            train_region = np.array(f['region']).transpose() 
            prob_idx = np.array(f['prob_idx']).transpose()
            
        print(f"✓ 原始数据形状: {train_data.shape}")
        print(f"✓ 原始标签形状: {train_region.shape}")
        
        # 分割数据集
        test_indices = np.where(prob_idx == 38)[0]
        train_val_indices = np.where(prob_idx != 38)[0]
        
        # 测试集
        X_test = train_data[test_indices, :]
        y_test = train_region[test_indices, :]
        
        # 训练+验证集
        train_val_data = train_data[train_val_indices, :]
        train_val_labels = train_region[train_val_indices, :]
        
        # 分层划分训练集和验证集
        X_train, X_val, y_train, y_val = train_test_split(
            train_val_data, train_val_labels,
            test_size=0.2,
            random_state=42,
            stratify=np.argmax(train_val_labels, axis=1)
        )
        
        print(f"\n📊 数据集划分:")
        print(f"   训练集: {X_train.shape[0]} 样本")
        print(f"   验证集: {X_val.shape[0]} 样本")
        print(f"   测试集: {X_test.shape[0]} 样本")
        
        # 标准化
        print(f"\n🔧 应用标准化...")
        self.scaler.fit(X_train)
        X_train_scaled = self.scaler.transform(X_train)
        X_val_scaled = self.scaler.transform(X_val)
        X_test_scaled = self.scaler.transform(X_test)
        
        # 验证标准化效果
        print(f"✓ 标准化验证:")
        print(f"   训练集均值: {np.mean(X_train_scaled):.6f} (应接近0)")
        print(f"   训练集标准差: {np.std(X_train_scaled):.6f} (应接近1)")
        
        # 为预训练模型准备分类特征
        if self.adapt_features:
            print(f"\n🔧 准备分类特征离散化...")
            # 使用KBinsDiscretizer将最后一个特征离散化为112个bins
            self.discretizer = KBinsDiscretizer(
                n_bins=112, 
                encode='ordinal', 
                strategy='quantile'
            )
            # 在训练集上fit
            self.discretizer.fit(X_train_scaled[:, -1:])
            print(f"✓ 离散化器已准备就绪")
        
        # 保存处理后的数据
        self.X_train = X_train_scaled
        self.X_val = X_val_scaled
        self.X_test = X_test_scaled
        self.y_train = y_train
        self.y_val = y_val
        self.y_test = y_test
        
        # 创建PyTorch数据集
        self.train_dataset = TensorDataset(
            torch.FloatTensor(X_train_scaled),
            torch.FloatTensor(y_train)
        )
        self.val_dataset = TensorDataset(
            torch.FloatTensor(X_val_scaled),
            torch.FloatTensor(y_val)
        )
        self.test_dataset = TensorDataset(
            torch.FloatTensor(X_test_scaled),
            torch.FloatTensor(y_test)
        )
        
        print(f"\n 数据预处理完成！")
        
    def train_dataloader(self):
        return DataLoader(
            self.train_dataset,
            batch_size=self.batch_size,
            shuffle=True,
            num_workers=self.num_workers,
            pin_memory=True,
            persistent_workers=True if self.num_workers > 0 else False
        )
    
    def val_dataloader(self):
        return DataLoader(
            self.val_dataset,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=self.num_workers,
            pin_memory=True,
            persistent_workers=True if self.num_workers > 0 else False
        )
    
    def test_dataloader(self):
        return DataLoader(
            self.test_dataset,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=self.num_workers,
            pin_memory=True,
            persistent_workers=True if self.num_workers > 0 else False
        )

# ============================================================================
# Part 2: TabNet模型定义 (支持entmax和分类特征)
# ============================================================================

class GLU_Block(nn.Module):
    """门控线性单元块"""
    def __init__(self, input_dim, output_dim, n_glu=2, first=False, shared_layers=None, virtual_batch_size=128, momentum=0.02):
        super().__init__()
        self.first = first
        self.shared_layers = shared_layers
        self.n_glu = n_glu
        self.glu_layers = nn.ModuleList()
        
        params = {
            'virtual_batch_size': virtual_batch_size,
            'momentum': momentum
        }
        
        # 决定使用shared layers还是独立layers
        fc = shared_layers[0] if shared_layers else None
        
        if shared_layers and not first:
            self.glu_layers.append(GLU_Layer(input_dim, output_dim, fc=fc, **params))
            for i in range(1, self.n_glu):
                fc = shared_layers[i] if i < len(shared_layers) else None
                self.glu_layers.append(GLU_Layer(output_dim, output_dim, fc=fc, **params))
        else:
            self.glu_layers.append(GLU_Layer(input_dim, output_dim, **params))
            for i in range(1, self.n_glu):
                self.glu_layers.append(GLU_Layer(output_dim, output_dim, **params))
    
    def forward(self, x):
        scale = torch.sqrt(torch.FloatTensor([0.5]).to(x.device))
        if self.first:
            x = self.glu_layers[0](x)
            layers_left = range(1, self.n_glu)
        else:
            layers_left = range(self.n_glu)
            
        for glu_id in layers_left:
            x = torch.add(x, self.glu_layers[glu_id](x))
            x = x * scale
        return x

class GLU_Layer(nn.Module):
    """单个GLU层"""
    def __init__(self, input_dim, output_dim, fc=None, virtual_batch_size=128, momentum=0.02):
        super().__init__()
        self.output_dim = output_dim
        if fc:
            self.fc = fc
        else:
            self.fc = nn.Linear(input_dim, 2 * output_dim, bias=False)
        initialize_non_glu(self.fc, input_dim, 2 * output_dim)
        
        self.bn = GBN(2 * output_dim, virtual_batch_size=virtual_batch_size, momentum=momentum)
    
    def forward(self, x):
        x = self.fc(x)
        x = self.bn(x)
        out = torch.mul(x[:, :self.output_dim], torch.sigmoid(x[:, self.output_dim:]))
        return out

class GBN(nn.Module):
    """
    Ghost Batch Normalization
    https://arxiv.org/abs/1705.08741
    """
    def __init__(self, input_dim, virtual_batch_size=128, momentum=0.02):
        super().__init__()
        self.input_dim = input_dim
        self.virtual_batch_size = virtual_batch_size
        self.bn = nn.BatchNorm1d(self.input_dim, momentum=momentum)
    
    def forward(self, x):
        chunks = x.chunk(int(np.ceil(x.shape[0] / self.virtual_batch_size)), 0)
        res = [self.bn(x_) for x_ in chunks]
        return torch.cat(res, dim=0)

class AttentiveTransformer(nn.Module):
    """注意力转换器 - 支持sparsemax和entmax"""
    def __init__(self, input_dim, output_dim, virtual_batch_size=128, momentum=0.02, mask_type="sparsemax"):
        super().__init__()
        self.fc = nn.Linear(input_dim, output_dim, bias=False)
        initialize_non_glu(self.fc, input_dim, output_dim)
        self.bn = GBN(output_dim, virtual_batch_size=virtual_batch_size, momentum=momentum)
        self.mask_type = mask_type
        
    def forward(self, priors, processed_feat):
        x = self.fc(processed_feat)
        x = self.bn(x)
        x = torch.mul(x, priors)
        
        if self.mask_type == "sparsemax":
            mask = sparsemax(x, dim=-1)
        elif self.mask_type == "entmax":
            mask = entmax15(x, dim=-1)
        else:
            mask = F.softmax(x, dim=-1)
            
        return mask

class TabNetEncoder(nn.Module):
    """TabNet编码器"""
    def __init__(
        self,
        input_dim,
        output_dim,
        n_d=8,
        n_a=8,
        n_steps=3,
        gamma=1.3,
        cat_idxs=[],
        cat_dims=[],
        cat_emb_dim=1,
        n_independent=2,
        n_shared=2,
        epsilon=1e-15,
        virtual_batch_size=128,
        momentum=0.02,
        mask_type="sparsemax",
    ):
        super().__init__()
        self.input_dim = input_dim
        self.output_dim = output_dim
        self.is_multi_task = isinstance(output_dim, list)
        self.n_d = n_d
        self.n_a = n_a
        self.n_steps = n_steps
        self.gamma = gamma
        self.epsilon = epsilon
        self.n_independent = n_independent
        self.n_shared = n_shared
        self.virtual_batch_size = virtual_batch_size
        self.mask_type = mask_type
        self.initial_splitter = FeatTransformer(
            self.input_dim,
            n_d + n_a,
            shared_layers=None,
            n_glu_independent=self.n_independent,
            virtual_batch_size=self.virtual_batch_size,
            momentum=momentum
        )
        
        # 创建shared layers
        if self.n_shared > 0:
            shared_feat_transform = []
            for i in range(self.n_shared):
                if i == 0:
                    shared_feat_transform.append(nn.Linear(self.input_dim, 2 * (n_d + n_a), bias=False))
                else:
                    shared_feat_transform.append(nn.Linear(n_d + n_a, 2 * (n_d + n_a), bias=False))
        else:
            shared_feat_transform = None
            
        self.shared = shared_feat_transform
        self.feat_transformers = nn.ModuleList()
        self.att_transformers = nn.ModuleList()
        
        # 创建decision steps
        for step in range(n_steps):
            transformer = FeatTransformer(
                self.input_dim,
                n_d + n_a,
                shared_layers=shared_feat_transform,
                n_glu_independent=self.n_independent,
                virtual_batch_size=self.virtual_batch_size,
                momentum=momentum,
            )
            attention = AttentiveTransformer(
                n_a,
                self.input_dim,
                virtual_batch_size=self.virtual_batch_size,
                momentum=momentum,
                mask_type=self.mask_type,
            )
            self.feat_transformers.append(transformer)
            self.att_transformers.append(attention)
    
    def forward(self, x, prior=None):
        x = self.initial_splitter(x)[:, self.n_d:]
        
        if prior is None:
            prior = torch.ones(x.shape).to(x.device)
            
        M_loss = 0
        att = x
        
        for step in range(self.n_steps):
            M = self.att_transformers[step](prior, att)
            M_loss += torch.mean(
                torch.sum(torch.mul(M, torch.log(M + self.epsilon)), dim=1)
            )
            
            prior = torch.mul(self.gamma - M, prior)
            
            masked_x = torch.mul(M, x)
            
            out = self.feat_transformers[step](masked_x)
            d = F.relu(out[:, :self.n_d])
            
            # 聚合步骤
            if step == 0:
                decision_out = d
            else:
                decision_out = torch.add(decision_out, d)
                
            # 更新attention
            att = out[:, self.n_d:]
            
        return decision_out, M_loss
    
    def forward_masks(self, x):
        """返回每个step的mask"""
        x = self.initial_splitter(x)[:, self.n_d:]
        
        prior = torch.ones(x.shape).to(x.device)
        M_explain = torch.zeros(x.shape).to(x.device)
        att = x
        masks = {}
        
        for step in range(self.n_steps):
            M = self.att_transformers[step](prior, att)
            masks[step] = M
            
            prior = torch.mul(self.gamma - M, prior)
            masked_x = torch.mul(M, x)
            out = self.feat_transformers[step](masked_x)
            
            # 聚合mask用于解释
            M_explain += M
            
            att = out[:, self.n_d:]
            
        return masks, M_explain

class TabNet(nn.Module):
    """完整的TabNet模型"""
    def __init__(
        self,
        input_dim,
        output_dim,
        n_d=8,
        n_a=8,
        n_steps=3,
        gamma=1.3,
        cat_idxs=[],
        cat_dims=[],
        cat_emb_dim=1,
        n_independent=2,
        n_shared=2,
        epsilon=1e-15,
        virtual_batch_size=128,
        momentum=0.02,
        mask_type="sparsemax",
    ):
        super().__init__()
        self.cat_idxs = cat_idxs or []
        self.cat_dims = cat_dims or []
        self.cat_emb_dim = cat_emb_dim
        
        self.input_dim = input_dim
        self.output_dim = output_dim
        self.n_d = n_d
        self.n_a = n_a
        self.n_steps = n_steps
        self.gamma = gamma
        self.epsilon = epsilon
        self.n_independent = n_independent
        self.n_shared = n_shared
        self.mask_type = mask_type
        
        # 处理分类特征的embedding
        if self.cat_idxs:
            self.cat_emb_dims = [min(cat_dim, cat_emb_dim) for cat_dim in self.cat_dims]
            self.embeddings = nn.ModuleList([
                nn.Embedding(cat_dim, emb_dim) 
                for cat_dim, emb_dim in zip(self.cat_dims, self.cat_emb_dims)
            ])
            self.post_embed_dim = self.input_dim + sum(self.cat_emb_dims) - len(self.cat_idxs)
        else:
            self.embeddings = None
            self.post_embed_dim = self.input_dim
            
        # 编码器
        self.encoder = TabNetEncoder(
            input_dim=self.post_embed_dim,
            output_dim=output_dim,
            n_d=n_d,
            n_a=n_a,
            n_steps=n_steps,
            gamma=gamma,
            n_independent=n_independent,
            n_shared=n_shared,
            epsilon=epsilon,
            virtual_batch_size=virtual_batch_size,
            momentum=momentum,
            mask_type=mask_type,
        )
        
        # 初始batch normalization
        self.initial_bn = nn.BatchNorm1d(self.post_embed_dim, momentum=momentum)
        
        # 最终映射层
        if self.output_dim != self.n_d:
            self.final_mapping = nn.Linear(self.n_d, self.output_dim, bias=False)
            initialize_non_glu(self.final_mapping, self.n_d, self.output_dim)
        else:
            self.final_mapping = None
    
    def forward(self, x):
        # 处理分类特征
        if self.embeddings is not None:
            cols = []
            cat_feat_counter = 0
            for feat_idx in range(self.input_dim):
                if feat_idx in self.cat_idxs:
                    # 这是分类特征
                    cat_idx = self.cat_idxs.index(feat_idx)
                    col = self.embeddings[cat_idx](x[:, feat_idx].long())
                    cols.append(col)
                    cat_feat_counter += 1
                else:
                    cols.append(x[:, feat_idx].unsqueeze(1))
            x = torch.cat(cols, dim=1)
            
        # Batch normalization
        x = self.initial_bn(x)
        
        # 编码
        output, M_loss = self.encoder(x)
        
        # 最终映射
        if self.final_mapping is not None:
            output = self.final_mapping(output)
            
        return output, M_loss
    
    def forward_masks(self, x):
        """获取注意力masks用于解释"""
        if self.embeddings is not None:
            cols = []
            for feat_idx in range(self.input_dim):
                if feat_idx in self.cat_idxs:
                    cat_idx = self.cat_idxs.index(feat_idx)
                    col = self.embeddings[cat_idx](x[:, feat_idx].long())
                    cols.append(col)
                else:
                    cols.append(x[:, feat_idx].unsqueeze(1))
            x = torch.cat(cols, dim=1)
            
        x = self.initial_bn(x)
        return self.encoder.forward_masks(x)

class FeatTransformer(nn.Module):
    """特征转换器"""
    def __init__(
        self,
        input_dim,
        output_dim,
        shared_layers,
        n_glu_independent,
        virtual_batch_size=128,
        momentum=0.02,
    ):
        super().__init__()
        params = {
            'n_glu': n_glu_independent,
            'virtual_batch_size': virtual_batch_size,
            'momentum': momentum
        }
        
        if shared_layers is None:
            self.shared = None
            self.fc = GLU_Block(input_dim, output_dim, first=True, **params)
        else:
            self.shared = GLU_Block(input_dim, output_dim, first=True, shared_layers=shared_layers, **params)
            self.fc = GLU_Block(output_dim, output_dim, **params)
    
    def forward(self, x):
        if self.shared is not None:
            x = self.shared(x)
        x = self.fc(x)
        return x

# ============================================================================
# Part 3: 辅助函数
# ============================================================================

def initialize_non_glu(module, input_dim, output_dim):
    """初始化非GLU层的权重"""
    gain_value = np.sqrt((input_dim + output_dim) / np.sqrt(4 * input_dim))
    torch.nn.init.xavier_normal_(module.weight, gain=gain_value)
    return

def initialize_glu(module, input_dim, output_dim):
    """初始化GLU层的权重"""
    gain_value = np.sqrt((input_dim + output_dim) / np.sqrt(input_dim))
    torch.nn.init.xavier_normal_(module.weight, gain=gain_value)
    return

def sparsemax(input, dim=-1):
    """Sparsemax activation function"""
    input = input - torch.max(input, dim=dim, keepdim=True)[0]
    zs = torch.sort(input, dim=dim, descending=True)[0]
    range_vals = torch.arange(1, input.size(dim) + 1).float().to(input.device)
    range_vals = range_vals.unsqueeze(0).expand_as(zs)
    
    bound = 1 + range_vals * zs
    cumsum_zs = torch.cumsum(zs, dim)
    is_gt = bound > cumsum_zs
    k = torch.max(is_gt * range_vals, dim, keepdim=True)[0]
    
    zs_sparse = is_gt * zs
    taus = (torch.sum(zs_sparse, dim, keepdim=True) - 1) / k
    taus = taus.expand_as(input)
    
    output = torch.max(torch.zeros_like(input), input - taus)
    return output

def entmax15(input, dim=-1):
    """
    Entmax 1.5 activation function
    基于 https://github.com/deep-spin/entmax
    """
    def _threshold_and_support(input, dim=-1):
        Xsrt, _ = torch.sort(input, descending=True, dim=dim)
        
        rho = torch.arange(1, input.size(dim) + 1).float().to(input.device)
        rho = rho.unsqueeze(0).expand_as(Xsrt)
        
        mean = Xsrt.cumsum(dim) / rho
        mean_sq = (Xsrt ** 2).cumsum(dim) / rho
        ss = rho * (mean_sq - mean ** 2)
        delta = (1 - ss) / rho
        
        delta_nz = torch.clamp(delta, 0)
        tau = mean - torch.sqrt(delta_nz)
        
        support_size = (tau <= Xsrt).sum(dim).unsqueeze(dim)
        tau_star = tau.gather(dim, support_size - 1)
        return tau_star, support_size
    
    def _entmax15(input, dim=-1):
        input = input / 2
        tau_star, _ = _threshold_and_support(input, dim)
        output = torch.clamp(input - tau_star, min=0) ** 2
        return output / output.sum(dim, keepdim=True)
    
    return _entmax15(input, dim)

# ============================================================================
# Part 4: Lightning模块 - 迁移学习实现
# ============================================================================

class TabNetTransferLearning(pl.LightningModule):
    """
    PyTorch Lightning模块 - 实现迁移学习
    """
    def __init__(self, 
                 pretrained_dir: str = 'tabnet_model_test0',
                 num_classes: int = 102,
                 learning_rate: float = 1e-3,
                 weight_decay: float = 1e-5,
                 freeze_stage: int = 0,
                 input_dim: int = 341):
        super().__init__()
        
        # 保存超参数
        self.save_hyperparameters()
        
        # 加载预训练模型配置
        with open(os.path.join(pretrained_dir, 'model_params.json'), 'r') as f:
            model_params = json.load(f)
        
        self.pretrained_params = model_params['init_params']
        
        # 输入适配层：341 -> 245
        self.input_adapter = nn.Sequential(
            nn.Linear(input_dim, self.pretrained_params['input_dim']),
            nn.BatchNorm1d(self.pretrained_params['input_dim']),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1)
        )
        
        # 创建与预训练模型完全匹配的TabNet
        self.tabnet = TabNet(
            input_dim=self.pretrained_params['input_dim'],
            output_dim=self.pretrained_params['n_d'],
            n_d=self.pretrained_params['n_d'],
            n_a=self.pretrained_params['n_a'],
            n_steps=self.pretrained_params['n_steps'],
            gamma=self.pretrained_params['gamma'],
            cat_idxs=self.pretrained_params['cat_idxs'],
            cat_dims=self.pretrained_params['cat_dims'],
            cat_emb_dim=self.pretrained_params['cat_emb_dim'][0],
            n_independent=self.pretrained_params['n_independent'],
            n_shared=self.pretrained_params['n_shared'],
            epsilon=self.pretrained_params['epsilon'],
            virtual_batch_size=128,
            momentum=self.pretrained_params['momentum'],
            mask_type=self.pretrained_params['mask_type'],
        )
        
        # 加载预训练权重
        self._load_pretrained_weights(os.path.join(pretrained_dir, 'network.pt'))
        
        # 任务特定的分类头
        self.classifier = nn.Sequential(
            nn.Linear(self.pretrained_params['n_d'], 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            
            nn.Linear(256, num_classes)
        )
        
        # 设置初始冻结状态
        self.freeze_stage = freeze_stage
        self.configure_freezing()
        
        # 记录指标
        self.epoch_metrics = {
            'train': {'loss': [], 'f1': []},
            'val': {'loss': [], 'f1': []},
            'test': {'loss': [], 'f1': []}
        }
        
    def _load_pretrained_weights(self, weight_path):
        """加载预训练权重"""
        print("\n📦 加载预训练权重...")
        checkpoint = torch.load(weight_path, map_location='cpu')
        
        # 加载TabNet权重
        try:
            incompatible_keys = self.tabnet.load_state_dict(checkpoint, strict=False)
            print(f"✓ 成功加载预训练TabNet权重")
            if incompatible_keys.missing_keys:
                print(f"  缺失的键: {len(incompatible_keys.missing_keys)}个")
            if incompatible_keys.unexpected_keys:
                print(f"  多余的键: {len(incompatible_keys.unexpected_keys)}个")
        except Exception as e:
            print(f"⚠️ 加载权重时出现问题: {e}")
            print("  将使用随机初始化的权重")
    
    def configure_freezing(self):
        """配置冻结策略"""
        print(f"\n🔧 配置冻结策略 - Stage {self.freeze_stage}")
        
        if self.freeze_stage == 0:
            # Stage 0: 只训练适配层和分类头
            for param in self.tabnet.parameters():
                param.requires_grad = False
            print("❄️ Stage 0: 冻结TabNet，只训练适配层和分类头")
            
        elif self.freeze_stage == 1:
            # Stage 1: 解冻最后的decision step
            for param in self.tabnet.parameters():
                param.requires_grad = False
                
            # 解冻最后一个step
            if hasattr(self.tabnet.encoder, 'feat_transformers'):
                last_idx = len(self.tabnet.encoder.feat_transformers) - 1
                for param in self.tabnet.encoder.feat_transformers[last_idx].parameters():
                    param.requires_grad = True
                for param in self.tabnet.encoder.att_transformers[last_idx].parameters():
                    param.requires_grad = True
                    
            # 解冻final mapping
            if self.tabnet.final_mapping is not None:
                for param in self.tabnet.final_mapping.parameters():
                    param.requires_grad = True
                    
            print("🔓 Stage 1: 解冻最后的decision step")
            
        elif self.freeze_stage == 2:
            # Stage 2: 解冻所有decision steps
            for param in self.tabnet.parameters():
                param.requires_grad = True
                
            # 保持BN层冻结
            for module in self.tabnet.modules():
                if isinstance(module, (nn.BatchNorm1d, GBN)):
                    for param in module.parameters():
                        param.requires_grad = False
                        
            print("🔓 Stage 2: 解冻所有decision steps（BN层除外）")
            
        else:  # Stage 3
            # Stage 3: 全部解冻
            for param in self.parameters():
                param.requires_grad = True
            print("🔥 Stage 3: 全部解冻 - 微调整个网络")
        
        # 确保适配层和分类头始终可训练
        for param in self.input_adapter.parameters():
            param.requires_grad = True
        for param in self.classifier.parameters():
            param.requires_grad = True
            
        # 统计参数
        total_params = sum(p.numel() for p in self.parameters())
        trainable_params = sum(p.numel() for p in self.parameters() if p.requires_grad)
        frozen_params = total_params - trainable_params
        
        print(f"\n📊 参数统计:")
        print(f"   总参数: {total_params:,}")
        print(f"   可训练: {trainable_params:,} ({100*trainable_params/total_params:.1f}%)")
        print(f"   冻结: {frozen_params:,} ({100*frozen_params/total_params:.1f}%)")
    
    def forward(self, x):
        # 适配输入维度
        x = self.input_adapter(x)
        
        # 处理分类特征（如果需要）
        if self.pretrained_params['cat_idxs'] and -1 in self.pretrained_params['cat_idxs']:
            # 最后一个特征需要离散化
            x_cont = x[:, :-1]
            x_cat = x[:, -1:]
            
            # 将连续值映射到[0, 111]的整数
            x_cat_discrete = torch.clamp(
                (x_cat * 56 + 56).long(),
                min=0,
                max=111
            )
            
            # 重新组合
            x = torch.cat([x_cont, x_cat_discrete.float()], dim=1)
        
        # TabNet编码
        encoded, M_loss = self.tabnet(x)
        
        # 分类
        logits = self.classifier(encoded)
        
        return logits, M_loss
    
    def _compute_loss_and_metrics(self, batch, stage='train'):
        """计算损失和Macro F1"""
        x, y = batch
        
        # 处理one-hot标签
        if y.dim() > 1 and y.shape[1] > 1:
            y_true = torch.argmax(y, dim=1)
        else:
            y_true = y.long()
        
        # 前向传播
        logits, M_loss = self(x)
        
        # 计算损失
        ce_loss = F.cross_entropy(logits, y_true)
        sparsity_loss = 1e-3 * M_loss
        total_loss = ce_loss + sparsity_loss
        
        # 计算Macro F1
        with torch.no_grad():
            y_pred = torch.argmax(logits, dim=1)
            macro_f1 = f1_score(
                y_true.cpu().numpy(), 
                y_pred.cpu().numpy(),
                average='macro',
                zero_division=0
            )
        
        return total_loss, macro_f1
    
    def training_step(self, batch, batch_idx):
        loss, macro_f1 = self._compute_loss_and_metrics(batch, 'train')
        
        self.log('train_loss', loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log('train_f1', macro_f1, on_step=False, on_epoch=True, prog_bar=True)
        
        return loss
    
    def validation_step(self, batch, batch_idx):
        loss, macro_f1 = self._compute_loss_and_metrics(batch, 'val')
        
        self.log('val_loss', loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log('val_f1', macro_f1, on_step=False, on_epoch=True, prog_bar=True)
        
        return {'val_loss': loss, 'val_f1': macro_f1}
    
    def test_step(self, batch, batch_idx):
        loss, macro_f1 = self._compute_loss_and_metrics(batch, 'test')
        
        self.log('test_loss', loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log('test_f1', macro_f1, on_step=False, on_epoch=True, prog_bar=True)
        
        return {'test_loss': loss, 'test_f1': macro_f1}
    
    def configure_optimizers(self):
        """配置优化器"""
        param_groups = []
        
        # TabNet参数（如果可训练）
        tabnet_params = [p for p in self.tabnet.parameters() if p.requires_grad]
        if tabnet_params:
            lr_scale = 0.1 ** max(0, 3 - self.freeze_stage)
            param_groups.append({
                'params': tabnet_params,
                'lr': self.hparams.learning_rate * lr_scale,
                'name': 'tabnet'
            })
        
        # 适配层参数
        param_groups.append({
            'params': self.input_adapter.parameters(),
            'lr': self.hparams.learning_rate,
            'name': 'adapter'
        })
        
        # 分类头参数
        param_groups.append({
            'params': self.classifier.parameters(),
            'lr': self.hparams.learning_rate,
            'name': 'classifier'
        })
        
        # 优化器
        optimizer = torch.optim.AdamW(
            param_groups,
            weight_decay=self.hparams.weight_decay
        )
        
        # 学习率调度器
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, 
            T_max=10,
            eta_min=1e-6
        )
        
        return {
            'optimizer': optimizer,
            'lr_scheduler': {
                'scheduler': scheduler,
                'interval': 'epoch',
                'frequency': 1,
                'monitor': 'val_f1'
            }
        }
    
    def on_train_epoch_end(self):
        """记录训练epoch结束时的指标"""
        metrics = self.trainer.logged_metrics
        if 'train_loss' in metrics:
            self.epoch_metrics['train']['loss'].append(metrics['train_loss'].item())
        if 'train_f1' in metrics:
            self.epoch_metrics['train']['f1'].append(metrics['train_f1'].item())
        
    def on_validation_epoch_end(self):
        """记录验证epoch结束时的指标"""
        metrics = self.trainer.logged_metrics
        if 'val_loss' in metrics:
            self.epoch_metrics['val']['loss'].append(metrics['val_loss'].item())
        if 'val_f1' in metrics:
            self.epoch_metrics['val']['f1'].append(metrics['val_f1'].item())

# ============================================================================
# Part 5: 回调函数
# ============================================================================

class ProgressiveUnfreezing(Callback):
    """逐步解冻回调"""
    def __init__(self, patience: int = 3, min_delta: float = 0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.wait = 0
        self.best_f1 = -float('inf')
        self.current_stage = 0
        
    def on_validation_epoch_end(self, trainer, pl_module):
        # 获取当前验证F1
        current_f1 = trainer.callback_metrics.get('val_f1', torch.tensor(0)).item()
        
        # 检查是否有改善
        if current_f1 > self.best_f1 + self.min_delta:
            self.best_f1 = current_f1
            self.wait = 0
        else:
            self.wait += 1
            
        print(f"\n📈 Epoch {trainer.current_epoch}: "
              f"Val F1 = {current_f1:.4f} (best = {self.best_f1:.4f}), "
              f"wait = {self.wait}/{self.patience}")
        
        # 判断是否进入下一阶段
        if self.wait >= self.patience and self.current_stage < 3:
            self.current_stage += 1
            self.wait = 0
            
            print(f"\n🚀 验证F1停滞 {self.patience} epochs，进入 Stage {self.current_stage}")
            
            # 更新模型的解冻策略
            pl_module.freeze_stage = self.current_stage
            pl_module.configure_freezing()
            
            # 重新配置优化器
            trainer.strategy.setup_optimizers(trainer)
            
            # 重置最佳F1
            self.best_f1 = current_f1

class TestMetricsCallback(Callback):
    """在每个epoch结束时计算测试集指标"""
    def __init__(self, test_dataloader):
        self.test_dataloader = test_dataloader
        
    def on_validation_epoch_end(self, trainer, pl_module):
        # 保存当前状态
        was_training = pl_module.training
        pl_module.eval()
        
        # 计算测试集指标
        test_losses = []
        test_f1_scores = []
        
        with torch.no_grad():
            for batch in self.test_dataloader:
                # 确保batch在正确的设备上
                if trainer.accelerator:
                    batch = trainer.accelerator.batch_to_device(batch)
                
                loss, f1 = pl_module._compute_loss_and_metrics(batch, 'test')
                test_losses.append(loss.item())
                test_f1_scores.append(f1)
        
        # 计算平均值
        test_loss_avg = np.mean(test_losses)
        test_f1_avg = np.mean(test_f1_scores)
        
        # 记录结果
        pl_module.epoch_metrics['test']['loss'].append(test_loss_avg)
        pl_module.epoch_metrics['test']['f1'].append(test_f1_avg)
        
        # 恢复训练状态
        if was_training:
            pl_module.train()
        
        # 获取其他指标
        train_f1 = trainer.callback_metrics.get('train_f1', torch.tensor(0)).item()
        val_f1 = trainer.callback_metrics.get('val_f1', torch.tensor(0)).item()
        
        # 打印结果
        print(f"\n📊 Epoch {trainer.current_epoch} - 所有数据集Macro F1:")
        print(f"   Train: {train_f1:.4f}")
        print(f"   Val:   {val_f1:.4f}")
        print(f"   Test:  {test_f1_avg:.4f}")

# ============================================================================
# Part 6: 主训练函数
# ============================================================================

def train_tabnet_transfer():
    """主训练函数"""
    
    # 配置
    config = {
        'data_path': '/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/DATA/TRAIN38_no_label43.mat',
        'pretrained_dir': 'tabnet_model_test0',
        'batch_size': 256,
        'num_epochs': 40,
        'learning_rate': 1e-3,
        'weight_decay': 1e-5,
        'patience': 3,
        'num_workers': 4,
    }
    
    print("="*80)
    print("🧠 TabNet迁移学习训练")
    print("="*80)
    print("\n📋 训练配置:")
    for key, value in config.items():
        print(f"   {key}: {value}")
    
    # 1. 准备数据
    data_module = BrainVoxelDataModule(
        data_path=config['data_path'],
        batch_size=config['batch_size'],
        num_workers=config['num_workers'],
        adapt_features=True
    )
    data_module.setup()
    
    # 2. 创建模型
    model = TabNetTransferLearning(
        pretrained_dir=config['pretrained_dir'],
        num_classes=102,
        learning_rate=config['learning_rate'],
        weight_decay=config['weight_decay'],
        freeze_stage=0,
        input_dim=341
    )
    
    # 3. 设置回调
    callbacks = [
        ProgressiveUnfreezing(patience=config['patience']),
        TestMetricsCallback(data_module.test_dataloader()),
        ModelCheckpoint(
            monitor='val_f1',
            mode='max',
            save_top_k=3,
            filename='tabnet-{epoch:02d}-{val_f1:.4f}',
            verbose=True
        ),
        EarlyStopping(
            monitor='val_f1',
            patience=10,
            mode='max',
            verbose=True
        ),
        LearningRateMonitor(logging_interval='epoch')
    ]
    
    # 4. 创建训练器
    trainer = pl.Trainer(
        max_epochs=config['num_epochs'],
        callbacks=callbacks,
        accelerator='gpu' if torch.cuda.is_available() else 'cpu',
        devices=1,
        precision=16,
        gradient_clip_val=1.0,
        deterministic=True,
        log_every_n_steps=50,
        enable_progress_bar=True,
        enable_checkpointing=True,
    )
    
    # 5. 训练
    print("\n🚀 开始训练...")
    print("="*80)
    
    trainer.fit(
        model,
        train_dataloaders=data_module.train_dataloader(),
        val_dataloaders=data_module.val_dataloader()
    )
    
    # 6. 最终测试
    print("\n🔍 最终测试...")
    test_results = trainer.test(model, dataloaders=data_module.test_dataloader())
    
    return model, model.epoch_metrics, test_results

# ============================================================================
# Part 7: 结果可视化
# ============================================================================

def plot_training_results(metrics_history):
    """绘制训练结果"""
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    
    epochs = range(len(metrics_history['train']['loss']))
    
    # 配色
    colors = {
        'train': '#1f77b4',
        'val': '#ff7f0e',
        'test': '#2ca02c'
    }
    
    # Loss曲线
    for idx, (split, ax) in enumerate(zip(['train', 'val', 'test'], axes[0])):
        if metrics_history[split]['loss']:
            ax.plot(range(len(metrics_history[split]['loss'])), 
                   metrics_history[split]['loss'], 
                   color=colors[split], linewidth=2.5, label=split.capitalize())
            ax.set_xlabel('Epoch', fontsize=12)
            ax.set_ylabel('Loss', fontsize=12)
            ax.set_title(f'{split.capitalize()} Loss', fontsize=14, fontweight='bold')
            ax.grid(True, alpha=0.3)
            ax.legend()
    
    # F1曲线
    for idx, (split, ax) in enumerate(zip(['train', 'val', 'test'], axes[1])):
        if metrics_history[split]['f1']:
            ax.plot(range(len(metrics_history[split]['f1'])), 
                   metrics_history[split]['f1'], 
                   color=colors[split], linewidth=2.5, label=split.capitalize())
            ax.set_xlabel('Epoch', fontsize=12)
            ax.set_ylabel('Macro F1', fontsize=12)
            ax.set_title(f'{split.capitalize()} Macro F1', fontsize=14, fontweight='bold')
            ax.grid(True, alpha=0.3)
            ax.legend()
            ax.set_ylim([0, 1])
    
    plt.suptitle('TabNet Transfer Learning Results', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig('tabnet_transfer_results.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    # 打印最终结果
    print("\n" + "="*80)
    print("📊 最终结果总结")
    print("="*80)
    
    for split in ['train', 'val', 'test']:
        if metrics_history[split]['f1']:
            final_f1 = metrics_history[split]['f1'][-1]
            initial_f1 = metrics_history[split]['f1'][0]
            improvement = final_f1 - initial_f1
            
            print(f"\n{split.upper()}集:")
            print(f"  最终 Macro F1: {final_f1:.4f}")
            print(f"  提升: {improvement:.4f} ({improvement/initial_f1*100:.1f}%)")

# ============================================================================
# 主执行入口
# ============================================================================

if __name__ == "__main__":
    # 训练模型
    model, metrics_history, test_results = train_tabnet_transfer()
    
    # 绘制结果
    plot_training_results(metrics_history)
    
    print("\n 训练完成！")